# ResNet1D XRF55 Phase — Kaggle Lite

**Model:** ResNet18 1D baseline — Phase (pre-DWT, z-score normalized)  
**Dataset:** XRF55 scene_01 — 11 actions, 30 subjects  
**Notebook:** Imports code from a Kaggle dataset; no inline code definitions.

---

## Required attached datasets

| Dataset name | Role | Mount path |
|---|---|---|
| `4-baselines-code` | Project source code | `/kaggle/input/datasets/imhoangt/4-baselines-code/` |
| `xrf55_processed_dataset` | Pre-processed XRF55 `.npy` files | `/kaggle/input/datasets/imhoangt/xrf55_processed_dataset/` |

In [ ]:
# Cell 1 — Verify packages (no extra installs needed)
# ResNet1D uses only torch, numpy, scikit-learn — pre-installed on Kaggle.
print('Packages OK — no extra installs needed.')

In [ ]:
# Cell 2 — Clone latest code from GitHub
import sys, subprocess
from pathlib import Path

CODE_PATH = Path('/kaggle/working/WaveConvMamba')
if not CODE_PATH.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/imhoangt/WaveConvMamba.git',
         str(CODE_PATH)],
        check=True
    )
else:
    print('Repo already cloned.')

sys.path.insert(0, str(CODE_PATH))
print(f'Code path  : {CODE_PATH}')

from baselines.resnet18_1d.resnet18_1d_cus.resnet1d_xrf55_phase.trainer import run_all
print('Import OK  : baselines.resnet18_1d.resnet18_1d_cus.resnet1d_xrf55_phase.trainer.run_all')

In [ ]:
# Cell 3 — Configuration (edit this)
# ─────────────────────────────────────────────────────────────
PROTOCOL = 'split'   # 'split' : reps 1-12 train, 13-14 val, 15-20 test (1 fold)
                     # 'loso'  : LOSO-5fold (5 folds, 6 subjects per fold)
N_SEEDS  = 1         # 1 -> seed=4 only  |  3 -> seeds [4, 8, 17]
# ─────────────────────────────────────────────────────────────

DATA_ROOT  = Path('/kaggle/input/datasets/imhoangt/xrf55_processed_dataset')
OUTPUT_DIR = Path('/kaggle/working/outputs/runs/resnet1d_xrf55_phase')

print(f'Protocol   : {PROTOCOL}')
print(f'N seeds    : {N_SEEDS}')
print(f'Data root  : {DATA_ROOT}')
print(f'Output dir : {OUTPUT_DIR}')

In [ ]:
# Cell 4 — Run training
# Resume-safe: if checkpoint.pt exists, training continues from it.
# Completed runs (result JSON exists) are skipped automatically.
summary = run_all(
    protocol   = PROTOCOL,
    n_seeds    = N_SEEDS,
    output_dir = OUTPUT_DIR,
    data_root  = DATA_ROOT,
)

In [ ]:
# Cell 5 — Results
import json

summary_path = OUTPUT_DIR / 'summary.json'
if summary_path.exists():
    with open(summary_path) as f:
        s = json.load(f)
    print(f"\n{'='*55}")
    print(f"  ResNet1D XRF55 Phase | {s['protocol'].upper()} | {s['n_runs']} runs")
    print(f"  Accuracy : {s['acc_mean']*100:.2f} +- {s['acc_std']*100:.2f}%")
    print(f"  F1 Macro : {s['f1_mean']*100:.2f} +- {s['f1_std']*100:.2f}%")
    print(f"{'='*55}")
    print()
    for key, r in s['runs'].items():
        print(f"  [{key}]  acc={r['test_accuracy']*100:.2f}%  f1={r['test_f1_macro']*100:.2f}%  "
              f"({r.get('total_time_s', '?')}s)")
    all_f1c = [r['test_f1_per_class'] for r in s['runs'].values()
               if 'test_f1_per_class' in r]
    if all_f1c:
        n = len(all_f1c[0])
        print(f"\n  Per-class F1 (mean across {len(all_f1c)} run(s)):")
        for i in range(n):
            mv = sum(run[i] for run in all_f1c) / len(all_f1c)
            print(f"    Class {i:2d}: {mv*100:.2f}%")
else:
    print('summary.json not found - training may not have completed.')

In [ ]:
# Cell 6 — Plots and ZIP archive
import csv, json, io, zipfile
import numpy as np
import matplotlib.pyplot as plt

_out         = OUTPUT_DIR
logs_dir     = _out / 'logs'
results_dir  = _out / 'results'
summary_path = _out / 'summary.json'

if not summary_path.exists():
    print('summary.json not found - run Cell 4 first.')
else:
    with open(summary_path) as f:
        s = json.load(f)

    run_curves = {}
    if logs_dir.exists():
        for csv_path in sorted(logs_dir.glob('*.csv')):
            rows = []
            with open(csv_path, newline='') as f:
                for row in csv.DictReader(f):
                    rows.append({k: float(v) for k, v in row.items()})
            if not rows:
                continue
            run_curves[csv_path.stem] = {
                'epochs': [r['epoch']      for r in rows],
                'losses': [r['train_loss'] for r in rows],
                'accs':   [r['test_acc']   for r in rows],
            }

    all_cms = []
    for r in s['runs'].values():
        if 'test_confusion_matrix' in r:
            all_cms.append(np.array(r['test_confusion_matrix']))

    n_cls      = all_cms[0].shape[0] if all_cms else 11
    cls_lbl    = [f'C{i}' for i in range(n_cls)]
    n_runs     = len(run_curves)
    n_total    = len(s['runs'])
    do_summary = n_total > 1
    colors     = plt.cm.tab10(np.linspace(0, 0.9, max(n_runs, 1)))

    if run_curves:
        max_ep   = max(len(c['epochs']) for c in run_curves.values())
        ep_range = list(range(1, max_ep + 1))
    else:
        max_ep, ep_range = 0, []

    def _save_buf(fig):
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=300, bbox_inches='tight')
        buf.seek(0)
        plt.show()
        plt.close(fig)
        return buf

    def _training_curve(epochs, losses, accs_pct, title):
        fig, ax = plt.subplots(figsize=(10, 5))
        ax2 = ax.twinx()
        ln1 = ax.plot( epochs, losses,   color='#E74C3C', linewidth=2, label='Loss')
        ln2 = ax2.plot(epochs, accs_pct, color='#2ECC71', linewidth=2, label='Test Acc (%)')
        ax.set_ylabel('Loss',          color='#E74C3C'); ax.tick_params( axis='y', labelcolor='#E74C3C')
        ax2.set_ylabel('Test Acc (%)', color='#2ECC71'); ax2.tick_params(axis='y', labelcolor='#2ECC71')
        ax2.set_ylim(0, 105)
        ax.set_xlabel('Epoch'); ax.set_title(title); ax.grid(True, alpha=0.3)
        lns = ln1 + ln2; ax.legend(lns, [l.get_label() for l in lns], loc='center right')
        fig.tight_layout()
        return fig

    def _confusion_matrix(cm_n, title):
        fig, ax = plt.subplots(figsize=(8, 7))
        im = ax.imshow(cm_n, cmap='Blues', vmin=0, vmax=1)
        ax.set_xticks(range(n_cls)); ax.set_xticklabels(cls_lbl, fontsize=7, rotation=45)
        ax.set_yticks(range(n_cls)); ax.set_yticklabels(cls_lbl, fontsize=7)
        ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        for i in range(n_cls):
            for j in range(n_cls):
                ax.text(j, i, f'{cm_n[i,j]:.2f}', ha='center', va='center', fontsize=6,
                        color='white' if cm_n[i, j] > 0.5 else 'black')
        fig.tight_layout()
        return fig

    run_bufs = {}
    for rk, r in s['runs'].items():
        bufs = {}
        acc_r = r['test_accuracy'] * 100
        f1_r  = r['test_f1_macro']  * 100

        c_r = run_curves.get(rk)
        if c_r:
            fig_tc = _training_curve(
                c_r['epochs'], c_r['losses'], [v*100 for v in c_r['accs']],
                title=f'ResNet1D XRF55 Phase [{rk}]  acc={acc_r:.2f}%  f1={f1_r:.2f}% — Training Curve'
            )
            bufs['tc'] = _save_buf(fig_tc)

        if 'test_confusion_matrix' in r:
            cm_r = np.array(r['test_confusion_matrix'])
            cm_n = cm_r.astype(float) / (cm_r.sum(axis=1, keepdims=True) + 1e-9)
            fig_cm = _confusion_matrix(
                cm_n,
                title=f'ResNet1D XRF55 Phase [{rk}]  acc={acc_r:.2f}%  f1={f1_r:.2f}% — Confusion Matrix'
            )
            bufs['cm'] = _save_buf(fig_cm)

        if bufs:
            run_bufs[rk] = bufs

    buf_stc = None
    buf_scm = None

    if do_summary:
        if run_curves:
            fig_s, ax_s = plt.subplots(figsize=(10, 5))
            ax2_s = ax_s.twinx()
            ln1_s, ln2_s = [], []
            if n_runs == 1:
                c = next(iter(run_curves.values()))
                ln1_s = ax_s.plot( c['epochs'], c['losses'],              color='#E74C3C', linewidth=2, label='Loss')
                ln2_s = ax2_s.plot(c['epochs'], [v*100 for v in c['accs']], color='#2ECC71', linewidth=2, label='Test Acc (%)')
            else:
                for (rk, c), col in zip(run_curves.items(), colors):
                    ax_s.plot( c['epochs'], c['losses'],              color='#E74C3C', alpha=0.2, linewidth=1)
                    ax2_s.plot(c['epochs'], [v*100 for v in c['accs']], color='#2ECC71', alpha=0.2, linewidth=1)
                ml  = np.mean([np.interp(ep_range, c['epochs'], c['losses']) for c in run_curves.values()], axis=0)
                ma  = np.mean([np.interp(ep_range, c['epochs'], c['accs'])   for c in run_curves.values()], axis=0)
                ln1_s = ax_s.plot( ep_range, ml,     color='#E74C3C', linewidth=2.5, label='Loss (mean)')
                ln2_s = ax2_s.plot(ep_range, ma*100, color='#2ECC71', linewidth=2.5, label='Test Acc (mean, %)')
            ax_s.set_ylabel('Loss',          color='#E74C3C'); ax_s.tick_params( axis='y', labelcolor='#E74C3C')
            ax2_s.set_ylabel('Test Acc (%)', color='#2ECC71'); ax2_s.tick_params(axis='y', labelcolor='#2ECC71')
            ax2_s.set_ylim(0, 105)
            ax_s.set_xlabel('Epoch'); ax_s.grid(True, alpha=0.3)
            ax_s.set_title(f'ResNet1D XRF55 Phase — {s["protocol"].upper()} ({n_total} runs) — Training Curve (summary)')
            if ln1_s and ln2_s:
                lns = ln1_s + ln2_s; ax_s.legend(lns, [l.get_label() for l in lns], loc='center right')
            fig_s.tight_layout()
            buf_stc = _save_buf(fig_s)

        if len(all_cms) > 1:
            cm_avg = np.mean([c / (c.sum(axis=1, keepdims=True) + 1e-9) for c in all_cms], axis=0)
            buf_scm = _save_buf(_confusion_matrix(
                cm_avg,
                title=f'ResNet1D XRF55 Phase — Confusion Matrix avg ({len(all_cms)} runs)'
            ))

    zip_path = _out / 'results_summary.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for rk, bufs in run_bufs.items():
            if 'tc' in bufs:
                zf.writestr(f'plots/{rk}_training_curve.png',   bufs['tc'].getvalue())
            if 'cm' in bufs:
                zf.writestr(f'plots/{rk}_confusion_matrix.png', bufs['cm'].getvalue())
        if buf_stc:
            zf.writestr('plots/summary_training_curve.png',   buf_stc.getvalue())
        if buf_scm:
            zf.writestr('plots/summary_confusion_matrix.png', buf_scm.getvalue())
        with open(summary_path, 'rb') as f:
            zf.writestr('summary.json', f.read())
        if results_dir.exists():
            for rp in sorted(results_dir.glob('*.json')):
                with open(rp, 'rb') as f:
                    zf.writestr(f'results/{rp.name}', f.read())
        if logs_dir.exists():
            for lp in sorted(logs_dir.glob('*.csv')):
                with open(lp, 'rb') as f:
                    zf.writestr(f'logs/{lp.name}', f.read())

    n_tc = sum(1 for b in run_bufs.values() if 'tc' in b)
    n_cm = sum(1 for b in run_bufs.values() if 'cm' in b)
    print(f'Saved ZIP: {zip_path}')
    print(f'  Per-run : {n_tc} training_curve  |  {n_cm} confusion_matrix')
    if do_summary:
        print(f'  Summary : {"training_curve" if buf_stc else "-"}  |  {"confusion_matrix" if buf_scm else "-"}')